
# Python Code Assistant

## _"Automate Execution, Comments, and Unit Tests for Python Code"_

### Key Features  
✅ **Execute the code and viualize results** – Instantly check the code functionality and time for excution.        
✅ **Auto-Generate Docstrings & Comments** – Instantly improve code clarity and maintainability.  
✅ **Unit Test Generation** – Ensure reliability with AI-generated test cases. 

### Imports

In [24]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

### Load OpenAI key into the environment

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print('API key looks fine')
else:
    print('Somehting is wrong with the API key.')

API key looks fine


### Initialize and define OpenAI model

In [33]:
openai = OpenAI()
OPENAI_MODEL = 'gpt-4o'

### Initialize system prompts

In [34]:
# Comment generator
system_prompt_comments = """
You are an AI model specializing in enhancing Python code documentation.
Generate detailed and precise docstrings and inline comments for the provided Python code.
Ensure the docstrings clearly describe the purpose, parameters, and return values of each function.
Inline comments should explain complex or non-obvious code segments.
Do not include any introductions, explanations, conclusions, or additional context.
The comments should be precise and as short as 1-2 sentences.
Return only the updated Python code enclosed within ```python ... ``` for proper formatting and syntax highlighting.
"""

# Unit Test generator 
system_prompt_unittests = """
You are an AI model specializing in generating comprehensive unit tests for Python code.
Create Python unit tests that thoroughly validate the functionality of the given code.
Use the `unittest` framework and ensure edge cases and error conditions are tested.
Do not include any comments, introductions, explanations, conclusions, or additional context.
Return only the unit test code enclosed within ```python ... ``` for proper formatting and syntax highlighting.
"""

### Initialize message prompts for the GPT model

In [35]:
def messages_for(task, python):
    if task == 'Comments':
        return [
            {"role": "system", "content": system_prompt_comments},
            {"role": "user", "content": f"""Generate comments for the following python code:
            {python}"""}
        ]
    else:
        return [
            {"role": "system", "content": system_prompt_unittests},
            {"role": "user", "content": f"""Generate unit tests for the following python code:
            {python}"""}
        ]

### Define easy, medium and hard level python programs

In [ ]:
# Easy level - reverse string

py_easy = """
import time

def reverse_string(s):
    return s[::-1]

if __name__ == "__main__":
    start_time = time.time()
    text = "Hello, World!"
    print(f"- Original string: {text}")
    print("- Reversed string:", reverse_string(text))
    execution_time = time.time() - start_time  
    print(f"\\n=> Execution Time: {execution_time:.6f} seconds")
"""

# Intermediate level - word frequency calculator

py_intermediate = """
import time

def word_frequency(text):
    words = text.lower().split()
    freq = {}
    for word in words:
        word = word.strip('.,!?')
        freq[word] = freq.get(word, 0) + 1
    return freq

if __name__ == "__main__":
    start_time = time.time()
    sentence = "Hello LLM engineers! Hello AI engineers!"
    frequencies = word_frequency(sentence)
    print("Word Frequencies:", frequencies)
    execution_time = time.time() - start_time  
    print(f"\\n=> Execution Time: {execution_time:.6f} seconds")
"""

# Hard level - infinite sequence of pseudorandom numbers

py_hard = """# Be careful to support large number sizes

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [38]:
exec(py_hard)

### Define streaming function for GPT

In [41]:
def stream_gpt(task, python):
    stream = openai.chat.completions.create(model=OPENAI_MODEL, 
                                            messages= messages_for(task, python),
                                            stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply

### Function to execute python program

In [39]:
def execute_python(code):
    try:
        output = io.StringIO()
        sys.stdout = output
        exec(code)
    finally:
        sys.stdout = sys.__stdout__
    return output.getvalue()

### Gradio UI

In [46]:
with gr.Blocks() as ui:
    with gr.Tab("Run"):
        with gr.Row():
            python = gr.Textbox(label="Python code:", value=py_hard, lines=15)
        with gr.Row():
            run = gr.Button("Run program")
        with gr.Row():
            python_out = gr.TextArea(label="Result:")
    with gr.Tab("Generate task"):
        with gr.Row():
            python = gr.Textbox(label="Python code:", value=py_hard, lines=50)
            task_out = gr.TextArea(label="Result:", lines=50)
        with gr.Row():
            option = gr.Dropdown(["Comments", "Unit Tests"], label="Select task", value="Comments")
        with gr.Row():
            generate = gr.Button("Generate task")
            

    run.click(execute_python, inputs=[python], outputs=[python_out])
    generate.click(stream_gpt, inputs=[option, python], outputs=[task_out])

ui.launch(inbrowser=True)